# Semester Project — Estimation of Obesity Levels
## Part 2 — Supervised Learning Baseline
### File 2: Decision Tree (Muhaimin Azeem)

**Group:** Muhaimin Azeem | Naeem Akmal | Rana Fahad Ahmad  
**Course:** Introduction to Machine Learning  

---
**This file covers:**
1. Train Decision Tree Classifier
2. Manual Gini Impurity calculation (hand-worked example)
3. Feature importance from Decision Tree


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate

drive.mount('/content/drive')
sns.set_style('whitegrid')
%matplotlib inline
print('Imports done ✓')

In [ ]:
X_train = np.load('/content/drive/MyDrive/X_train_processed.npy')
X_test  = np.load('/content/drive/MyDrive/X_test_processed.npy')
y_train = np.load('/content/drive/MyDrive/y_train.npy', allow_pickle=True)
y_test  = np.load('/content/drive/MyDrive/y_test.npy',  allow_pickle=True)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

## 1. Decision Tree — Theory

A Decision Tree splits data recursively by choosing the feature + threshold that **best reduces impurity**.

**Gini Impurity** (used by scikit-learn's CART algorithm):

$$\text{Gini}(S) = 1 - \sum_{c} p_c^2$$

A node is **pure** (Gini = 0) if all samples belong to one class.  
The best split minimises the **weighted Gini** of the two child nodes.

**Advantages over Naive Bayes:**
- No independence assumption
- Naturally handles feature interactions
- Interpretable (you can print the tree)

**Disadvantages:**
- Prone to overfitting → needs max_depth or min_samples_leaf tuning
- Unstable: small data changes → very different tree

In [ ]:
# ── Manual Gini Impurity: family_history_with_overweight ─────────────────
df_raw = pd.read_csv('/content/drive/MyDrive/ObesityDataSet_raw_and_data_sinthetic.csv')

sample = df_raw[['family_history_with_overweight', 'NObeyesdad']].sample(n=10, random_state=7).reset_index(drop=True)
print('Sample (10 rows):')
print(sample.to_string())

def gini(labels):
    """Gini impurity."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = pd.Series(labels).value_counts()
    probs  = counts / n
    return float(1 - np.sum(probs ** 2))

# Step 1
G_S = gini(sample['NObeyesdad'])
print(f'\nStep 1 — Gini of full sample  Gini(S) = {G_S:.4f}')

# Step 2
feat = 'family_history_with_overweight'
print(f'\nStep 2 — Split on {feat}:')
weighted_G = 0.0
for val in sample[feat].unique():
    sub = sample[sample[feat] == val]['NObeyesdad']
    g   = gini(sub)
    w   = len(sub) / len(sample)
    weighted_G += w * g
    print(f'  {feat}={val}: n={len(sub)}, Gini={g:.4f}, weight={w:.2f}')

print(f'\n  Weighted Gini = {weighted_G:.4f}')

# Step 3 – Gini Gain
GG = G_S - weighted_G
print(f'\nStep 3 — Gini Gain = {G_S:.4f} - {weighted_G:.4f} = {GG:.4f}')
print('Higher Gini Gain → better split.')

In [ ]:
# ── Train Decision Tree ────────────────────────────────────────────────────
dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_f1  = f1_score(y_test, y_pred_dt, average='macro')

print(f'Decision Tree  |  Accuracy: {dt_acc:.4f}  |  Macro F1: {dt_f1:.4f}')
print()
print(classification_report(y_test, y_pred_dt))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dt,
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)
ax.set_title('Decision Tree — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────
# Build feature name list (must match Part 1 pipeline order)
num_feat_names = ['Age','Height','Weight','FCVC','NCP','CH2O','FAF','TUE','BMI']

# Reconstruct OHE category names from raw data
cat_cols_order = ['Gender','family_history_with_overweight','FAVC',
                  'CAEC','SMOKE','SCC','CALC','MTRANS']
cat_feat_names = []
for col in cat_cols_order:
    for val in sorted(df_raw[col].unique()):
        cat_feat_names.append(f'{col}_{val}')

all_feat_names = num_feat_names + cat_feat_names

# Trim/pad to match processed array width
n_feats = X_train.shape[1]
if len(all_feat_names) > n_feats:
    all_feat_names = all_feat_names[:n_feats]
elif len(all_feat_names) < n_feats:
    all_feat_names += [f'feat_{i}' for i in range(len(all_feat_names), n_feats)]

imp_df = pd.DataFrame({'feature': all_feat_names,
                       'importance': dt.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(8, 5))
sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
plt.title('Top-15 Feature Importances (Decision Tree — Gini)')
plt.xlabel('Mean Decrease in Gini Impurity')
plt.tight_layout()
plt.show()

In [ ]:
import pickle

with open('/content/drive/MyDrive/dt_model.pkl', 'wb') as f:
    pickle.dump(dt, f)

np.save('/content/drive/MyDrive/y_pred_dt.npy', y_pred_dt)

print('Decision Tree model saved to Drive ✓')
print(f'Final  →  Accuracy: {dt_acc:.4f}  |  Macro F1: {dt_f1:.4f}')